In [30]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
LLM=ChatGoogleGenerativeAI(
    model='gemini-3.5-flash-lite')

In [12]:
 !pip install pypdf

  Using cached pypdf-6.15.0-py3-none-any.whl.metadata (7.5 kB)
Using cached pypdf-6.15.0-py3-none-any.whl (378 kB)


In [13]:
loader = PyPDFLoader("My_AxiOraa_Ltd_Synthetic_Annual_Report_2025.pdf")
documents = loader.load()

report=""
for page in documents:
    report += page.page_content+"\n"

print("Total pages:", len(documents))
print(report[:1000])

Total pages: 9
AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [15]:
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

In [17]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 2
)

chunks = text_spliter.split_documents(documents)

In [21]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/16.3 MB 10.1 MB/s eta 0:00:02
   --------- ------------------------------ 3.7/16.3 MB 9.6 MB/s eta 0:00:02
   -------------- ------------------------- 5.8/16.3 MB 9.7 MB/s eta 0:00:02
   ------------------ --------------------- 7.6/16.3 MB 9.5 MB/s eta 0:00:01
   ------------------------- -------------- 10.2/16.3 MB 9.9 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.3 MB 10.1 MB/s eta 0:00:01
   ------------------------------------ --- 14.7/16.3 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------  16.0/16.3 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 9.0 MB/s eta 0:00:00


In [22]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)

In [23]:
query = "What investments did AxiOraa make in AI?"

In [25]:
result = vector_store.similarity_search_with_score(
    query, k=5
)

for i, (doc, score) in enumerate (result, start=1):
    print(f"result{i}")
    print(f"similarity_score:{score}")
    print(f"content:{doc.page_content[:1000]}")

result1
similarity_score:0.5348026752471924
content:AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observation

In [35]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs= {"k":2}
)


In [36]:
rag_prompt = ChatPromptTemplate.from_template(
    """
        You are a Senior Consultant and your job is to answer the user's questions only by using the retrieved context.

        If the answer cannot be found in the contex, simple say - "I couldn't find this information"

        Retrieved contex:{context}

        Question: {question}

        provide:
        1. Answer
        2. Support Evidence 
        3. Page Number
    """
)

In [37]:
rag_chain = (
    rag_prompt | LLM | StrOutputParser()
)

In [38]:
question="Summarize the AI Investments made by AxiOraa"

retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response=rag_chain.invoke(
    {"context":context,
     "question":question}
)
print(response)

1. **Answer:** 
AxiOraa made significant investments in LLMOps, vector databases, evaluation frameworks, LangChain accelerators, and AI observability.

2. **Support Evidence:** 
"Significant investments were made in LLMOps, vector databases, evaluation frameworks, LangChain accelerators and AI observability."

3. **Page Number:** 
I couldn't find this information
